# Run the HIF / CICIDS2017 comparison on Google Colab

This notebook clones the repository, installs the pinned dependencies,
downloads the dataset and runs the full comparison. It is meant to run on
Colab so you do not have to run the heavy pipeline on your own machine.

Recommended: Runtime -> Change runtime type -> High-RAM (the dataset has
millions of rows). A GPU is not needed; the models run on CPU.

## 1. Clone the repository
Edit REPO_URL if you forked it or pushed it elsewhere.

In [ ]:
REPO_URL = "https://github.com/Dicotomico23/hif-cicids2017.git"
import os
repo = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.isdir(repo):
    !git clone $REPO_URL
%cd $repo
!git log --oneline -1

## 2. Install dependencies

In [ ]:
# Colab already ships numpy/pandas/scipy/scikit-learn/matplotlib. Install only
# the missing pieces and the package itself, without forcing version changes.
!pip install -q -U kagglehub imbalanced-learn optuna
!pip install -q -e . --no-deps
# Note: this uses Colab's library versions. For the exact pinned environment
# used in the paper, install requirements.txt locally instead.


## 3. Get the dataset

Primary source: the archived copy configured in `data/download.py`
(Zenodo / GitHub Release). No Kaggle account is needed once it is set.

Kaggle is only an emergency fallback and is disabled by default. The
optional block at the bottom of the next cell enables it if the archive
is not configured yet.

In [ ]:
# Primary: fetch the archived dataset (Zenodo / GitHub Release).
!python data/download.py

# -------------------------------------------------------------------------
# EMERGENCY FALLBACK (optional). Only if the archive is not configured yet.
# Kaggle needs a token; uncomment the lines below to use it.
#
# from google.colab import files
# files.upload()  # select your kaggle.json
# import os
# os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
# !mv kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
# !ALLOW_KAGGLE=1 python data/download.py


## 4. Run the full comparison

Writes the results table and figures to `results/`. To do a quick partial
run first, add `--nrows 100000`.

In [ ]:
!python reproduce/run_comparison.py --output results --optimize --n_trials 20

## 5. Show the results

In [ ]:
import pandas as pd
from IPython.display import Image, display
print(pd.read_csv('results/table5_results.csv').to_string(index=False))
for fig in ['fig_radar_metrics', 'fig_hif_confusion_matrix', 'fig_bar_balanced_acc', 'fig_bar_precision']:
    display(Image('results/figures/%s.png' % fig))